# Premier League 2026/27 Score Predictions

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RivaroFarrelino/epl-prediction-mw6-2026-27/blob/main/notebooks/epl_score_prediction.ipynb)

Predicts Premier League scorelines with a Dixon-Coles model. The model is fitted on the last two seasons (2024/25 and 2025/26) plus whatever has been played so far this season, with older matches counting for less.

Data, all free:
- [football-data.co.uk](https://www.football-data.co.uk) for results and closing odds, Premier League (`E0`) and Championship (`E1`)
- The [Fantasy Premier League API](https://fantasy.premierleague.com/api/fixtures/) for fixtures and kickoff times

To run it, set the config below and use Runtime > Run all. The last cell downloads the outputs as a zip.

## 1. Config

| Variable | What it does |
|---|---|
| `MATCHWEEK` | Matchweek to predict |
| `XI` | Time decay per day. 0.0019 is a half-life of about a year. Higher means recent games count for more |
| `L2` | Regularization. Keeps teams with little data from getting extreme ratings |
| `RUN_BACKTEST` | Test the model on the second half of 2025/26 and compare it with the bookmakers |
| `USE_DRIVE` | Keep the data cache and outputs in Google Drive so they survive a runtime reset |
| `REFRESH` | Re-download every CSV |

In [ ]:
TRAIN_SEASONS = ["2425", "2526"]
CURRENT_SEASON = "2627"
LEAGUES = ["E0", "E1"]
MATCHWEEK = 6

XI = 0.0019
L2 = 1.0

RUN_BACKTEST = True
BT_START = "2026-01-01"
BT_END = "2026-06-01"

REFRESH = False
USE_DRIVE = False
DRIVE_DIR = "/content/drive/MyDrive/epl_predictions"
DEBUG = False

## 2. Setup
Everything used here (numpy, pandas, scipy, requests, plotly) comes preinstalled on Colab.

In [ ]:
import logging
import shutil
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import requests
from IPython.display import display
from scipy.optimize import minimize
from scipy.special import gammaln
from scipy.stats import poisson

BASE_URL = "https://www.football-data.co.uk/mmz4281/{season}/{league}.csv"
FPL_FIXTURES_URL = "https://fantasy.premierleague.com/api/fixtures/"
FPL_BOOTSTRAP_URL = "https://fantasy.premierleague.com/api/bootstrap-static/"
HEADERS = {"User-Agent": "Mozilla/5.0 (epl-predictor)"}
MAX_GOALS = 10
RETRIES = 5
RETRY_WAIT = 5
STALE_SECONDS = 12 * 3600
TZ = "Asia/Jakarta"

FPL_TO_FD = {
    "Man Utd": "Man United",
    "Spurs": "Tottenham",
    "Nott'm Forest": "Nott'm Forest",
}

FALLBACK_FIXTURES = {
    6: [
        ("Arsenal", "Leeds"),
        ("Aston Villa", "Brentford"),
        ("Chelsea", "Bournemouth"),
        ("Ipswich", "Fulham"),
        ("Sunderland", "Brighton"),
        ("Man United", "Tottenham"),
        ("Crystal Palace", "Nott'm Forest"),
        ("Hull", "Everton"),
        ("Liverpool", "Man City"),
        ("Coventry", "Newcastle"),
    ]
}

ODDS_PRIORITY = [
    ("PSCH", "PSCD", "PSCA"),
    ("AvgCH", "AvgCD", "AvgCA"),
    ("B365CH", "B365CD", "B365CA"),
    ("PSH", "PSD", "PSA"),
    ("AvgH", "AvgD", "AvgA"),
    ("B365H", "B365D", "B365A"),
]

log = logging.getLogger("epl")
log.handlers.clear()
_handler = logging.StreamHandler(sys.stdout)
_handler.setFormatter(logging.Formatter("%(asctime)s | %(levelname)-7s | %(message)s", "%H:%M:%S"))
log.addHandler(_handler)
log.setLevel(logging.DEBUG if DEBUG else logging.INFO)
log.propagate = False

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_DIR = Path(DRIVE_DIR)
else:
    BASE_DIR = Path("epl_predictions")
DATA_DIR = BASE_DIR / "data"
OUT_DIR = BASE_DIR / "output"
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)
log.info("Working directory: %s", BASE_DIR.resolve())

## 3. Load data
CSVs are cached in `data/`. The current season file gets refreshed once it's more than 12 hours old. Failed requests are retried 5 times, 5 seconds apart.

In [ ]:
def http_get(url):
    for attempt in range(1, RETRIES + 1):
        try:
            r = requests.get(url, headers=HEADERS, timeout=30)
            r.raise_for_status()
            log.debug("GET %s -> %d (%d bytes)", url, r.status_code, len(r.content))
            return r
        except requests.RequestException as e:
            log.warning("GET failed %s (attempt %d/%d): %s", url, attempt, RETRIES, e)
            if attempt == RETRIES:
                raise
            time.sleep(RETRY_WAIT)


def load_season(league, season):
    path = DATA_DIR / f"{league}_{season}.csv"
    stale = season == CURRENT_SEASON and path.exists() and time.time() - path.stat().st_mtime > STALE_SECONDS
    if REFRESH or stale or not path.exists():
        url = BASE_URL.format(season=season, league=league)
        try:
            path.write_bytes(http_get(url).content)
            log.info("Downloaded %s", url)
        except requests.RequestException:
            if not path.exists():
                raise
            log.warning("Download failed, using old cache %s", path)
    else:
        log.info("Using cache %s", path.name)

    raw = pd.read_csv(path, encoding="latin-1", on_bad_lines="skip")
    raw = raw.dropna(subset=["HomeTeam", "AwayTeam", "FTHG", "FTAG"])
    df = pd.DataFrame(
        {
            "Date": pd.to_datetime(raw["Date"], dayfirst=True, format="mixed"),
            "HomeTeam": raw["HomeTeam"].str.strip(),
            "AwayTeam": raw["AwayTeam"].str.strip(),
            "FTHG": raw["FTHG"].astype(int),
            "FTAG": raw["FTAG"].astype(int),
            "League": league,
            "Season": season,
        }
    )
    for side, i in (("OddsH", 0), ("OddsD", 1), ("OddsA", 2)):
        col = pd.Series(np.nan, index=raw.index)
        for triple in ODDS_PRIORITY:
            if all(c in raw.columns for c in triple):
                col = col.fillna(pd.to_numeric(raw[triple[i]], errors="coerce"))
        df[side] = col.to_numpy()
    log.info("%s %s: %d matches (%s to %s)", league, season, len(df), df.Date.min().date(), df.Date.max().date())
    return df


matches = pd.concat(
    [load_season(lg, s) for s in TRAIN_SEASONS + [CURRENT_SEASON] for lg in LEAGUES],
    ignore_index=True,
).sort_values("Date", ignore_index=True)
log.info("Total: %d matches, %d teams", len(matches), len(set(matches.HomeTeam) | set(matches.AwayTeam)))
matches.tail()

## 4. Model

For home team *i* against away team *j*:

- home goals ~ Poisson(λ), where log λ = μ + home + attack<sub>i</sub> + defence<sub>j</sub>
- away goals ~ Poisson(ν), where log ν = μ + attack<sub>j</sub> + defence<sub>i</sub>
- ρ adjusts the odds of 0-0, 1-0, 0-1 and 1-1, which a plain Poisson model gets slightly wrong
- each match is weighted by exp(−XI × days since the match)
- Championship matches get their own offset. Teams that moved between the two divisions tie the leagues together, which is how promoted sides like Coventry and Hull end up with a sensible rating

Fitting uses L-BFGS-B with an analytic gradient, so it only takes a moment even with a few thousand matches.

In [ ]:
class DixonColes:
    def __init__(self, xi=XI, l2=L2):
        self.xi = xi
        self.l2 = l2

    def fit(self, df, ref_date, init=None):
        t0 = time.time()
        self.teams = sorted(set(df.HomeTeam) | set(df.AwayTeam))
        self.idx = {t: i for i, t in enumerate(self.teams)}
        n = len(self.teams)
        h = df.HomeTeam.map(self.idx).to_numpy()
        a = df.AwayTeam.map(self.idx).to_numpy()
        x = df.FTHG.to_numpy(dtype=float)
        y = df.FTAG.to_numpy(dtype=float)
        champ = (df.League == "E1").to_numpy(dtype=float)
        days = (pd.Timestamp(ref_date) - df.Date).dt.days.clip(lower=0).to_numpy(dtype=float)
        w = np.exp(-self.xi * days)
        m = [(x == 0) & (y == 0), (x == 0) & (y == 1), (x == 1) & (y == 0), (x == 1) & (y == 1)]
        lgf = gammaln(x + 1) + gammaln(y + 1)
        l2 = self.l2

        def objective(p):
            att, dfn = p[:n], p[n:2 * n]
            home, mu0, off, rho = p[2 * n:]
            ll = mu0 + home + att[h] + dfn[a] + off * champ
            lm = mu0 + att[a] + dfn[h] + off * champ
            lam, mu = np.exp(ll), np.exp(lm)
            tau = np.select(m, [1 - lam * mu * rho, 1 + lam * rho, 1 + mu * rho, np.full_like(lam, 1 - rho)], 1.0)
            tau = np.maximum(tau, 1e-10)
            logl = np.log(tau) + x * ll - lam + y * lm - mu - lgf
            nll = -(w * logl).sum() + l2 * (att @ att + dfn @ dfn)

            d00 = -lam * mu * rho / tau
            dll = x - lam + np.select([m[0], m[1]], [d00, lam * rho / tau], 0.0)
            dlm = y - mu + np.select([m[0], m[2]], [d00, mu * rho / tau], 0.0)
            drho = np.select(m, [-lam * mu / tau, lam / tau, mu / tau, -1 / tau], 0.0)
            gl, gm = w * dll, w * dlm
            g_att = np.bincount(h, gl, n) + np.bincount(a, gm, n)
            g_def = np.bincount(a, gl, n) + np.bincount(h, gm, n)
            grad = -np.concatenate([g_att, g_def, [gl.sum(), gl.sum() + gm.sum(), ((gl + gm) * champ).sum(), (w * drho).sum()]])
            grad[:n] += 2 * l2 * att
            grad[n:2 * n] += 2 * l2 * dfn
            return nll, grad

        p0 = np.zeros(2 * n + 4)
        p0[2 * n:] = [0.25, np.log(max((x.mean() + y.mean()) / 2, 0.1)), 0.0, 0.0]
        if init is not None:
            for t, i in self.idx.items():
                if t in init.idx:
                    p0[i], p0[n + i] = init.att[init.idx[t]], init.dfn[init.idx[t]]
            p0[2 * n:] = [init.home, init.mu0, init.off, init.rho]

        bounds = [(None, None)] * (2 * n + 3) + [(-0.25, 0.25)]
        res = minimize(objective, p0, jac=True, method="L-BFGS-B", bounds=bounds, options={"maxiter": 3000})
        if not res.success:
            log.warning("Optimizer did not converge: %s", res.message)
        p = res.x
        self.att, self.dfn = p[:n], p[n:2 * n]
        self.home, self.mu0, self.off, self.rho = p[2 * n:]
        log.debug(
            "Fit %d matches, %d teams, %d iterations, %.2fs | home=%.3f rho=%.3f offset_E1=%.3f",
            len(df), n, res.nit, time.time() - t0, self.home, self.rho, self.off,
        )
        return self

    def score_matrix(self, home, away):
        i, j = self.idx[home], self.idx[away]
        lam = np.exp(self.mu0 + self.home + self.att[i] + self.dfn[j])
        mu = np.exp(self.mu0 + self.att[j] + self.dfn[i])
        g = np.arange(MAX_GOALS + 1)
        mat = np.outer(poisson.pmf(g, lam), poisson.pmf(g, mu))
        mat[0, 0] *= 1 - lam * mu * self.rho
        mat[0, 1] *= 1 + lam * self.rho
        mat[1, 0] *= 1 + mu * self.rho
        mat[1, 1] *= 1 - self.rho
        return mat / mat.sum(), lam, mu

    def ratings(self):
        return pd.DataFrame(
            {"Team": self.teams, "Attack": np.exp(self.att), "Defence": np.exp(self.dfn)}
        ).assign(Rating=lambda d: d.Attack / d.Defence).sort_values("Rating", ascending=False, ignore_index=True)


def outcome_probs(mat):
    return np.tril(mat, -1).sum(), np.trace(mat), np.triu(mat, 1).sum()


def summarize(mat, lam, mu):
    ph, pd_, pa = outcome_probs(mat)
    flat = np.argsort(mat, axis=None)[::-1][:3]
    top = [f"{i}-{j} ({mat[i, j]:.1%})" for i, j in zip(*np.unravel_index(flat, mat.shape))]
    g = np.arange(MAX_GOALS + 1)
    total = g[:, None] + g[None, :]
    return {
        "xG_Home": lam,
        "xG_Away": mu,
        "P_Home": ph,
        "P_Draw": pd_,
        "P_Away": pa,
        "Most_Likely": top[0].split(" ")[0],
        "Top3_Scores": " | ".join(top),
        "Over2.5": mat[total >= 3].sum(),
        "BTTS": mat[1:, 1:].sum(),
    }

## 5. Backtest (optional)
Goes through the second half of 2025/26 one week at a time. Each week the model is refitted on everything before that week, then predicts that week's games. The results are compared with the bookmakers' closing odds with the margin removed.

Lower log loss and RPS is better. Getting close to the bookmakers is already a decent result. Beating them consistently is rare.

In [ ]:
def rps(p, o):
    cp, co = np.cumsum(p, axis=1)[:, :2], np.cumsum(o, axis=1)[:, :2]
    return ((cp - co) ** 2).sum(axis=1) / 2


def backtest(df, start, end):
    test = df[(df.League == "E0") & (df.Date >= start) & (df.Date < end)]
    if test.empty:
        log.error("No test matches between %s and %s", start, end)
        return None
    log.info("Backtesting %d EPL matches (%s to %s), refitting weekly", len(test), start, end)
    model, rows = None, []
    for _, block in test.groupby(test.Date.dt.to_period("W")):
        cutoff = block.Date.min()
        model = DixonColes().fit(df[df.Date < cutoff], cutoff, init=model)
        for r in block.itertuples(index=False):
            if r.HomeTeam not in model.idx or r.AwayTeam not in model.idx:
                log.debug("Skipping %s vs %s, team not in training data yet", r.HomeTeam, r.AwayTeam)
                continue
            mat, _, _ = model.score_matrix(r.HomeTeam, r.AwayTeam)
            i, j = np.unravel_index(mat.argmax(), mat.shape)
            ph, pd_, pa = outcome_probs(mat)
            rows.append(
                {
                    "Date": r.Date.date(), "Home": r.HomeTeam, "Away": r.AwayTeam,
                    "Score": f"{r.FTHG}-{r.FTAG}", "Predicted": f"{i}-{j}",
                    "pH": ph, "pD": pd_, "pA": pa,
                    "res": 0 if r.FTHG > r.FTAG else 1 if r.FTHG == r.FTAG else 2,
                    "OddsH": r.OddsH, "OddsD": r.OddsD, "OddsA": r.OddsA,
                }
            )
        log.debug("Week of %s done, %d matches so far", cutoff.date(), len(rows))

    bt = pd.DataFrame(rows)
    onehot = np.eye(3)[bt.res]
    pm = bt[["pH", "pD", "pA"]].to_numpy()
    report = {
        "Matches": len(bt),
        "1X2 accuracy": (pm.argmax(1) == bt.res).mean(),
        "Exact score accuracy": (bt.Score == bt.Predicted).mean(),
        "Log loss (model)": -np.log(pm[np.arange(len(bt)), bt.res]).mean(),
        "RPS (model)": rps(pm, onehot).mean(),
    }
    has_odds = bt[["OddsH", "OddsD", "OddsA"]].notna().all(axis=1).to_numpy()
    if has_odds.any():
        inv = 1 / bt.loc[has_odds, ["OddsH", "OddsD", "OddsA"]].to_numpy()
        pb = inv / inv.sum(axis=1, keepdims=True)
        res_b = bt.res.to_numpy()[has_odds]
        report["Log loss (model, matches with odds)"] = -np.log(pm[has_odds][np.arange(has_odds.sum()), res_b]).mean()
        report["Log loss (bookmakers)"] = -np.log(pb[np.arange(len(pb)), res_b]).mean()
        report["RPS (bookmakers)"] = rps(pb, onehot[has_odds]).mean()

    bt.drop(columns="res").round(3).to_csv(OUT_DIR / "backtest_detail.csv", index=False)
    log.info("Backtest detail saved to %s", OUT_DIR / "backtest_detail.csv")
    return pd.Series(report, name="Value").to_frame()


if RUN_BACKTEST:
    t0 = time.time()
    bt_report = backtest(matches, BT_START, BT_END)
    log.info("Backtest finished in %.1fs", time.time() - t0)
    if bt_report is not None:
        display(bt_report.style.format(lambda v: f"{v:,.0f}" if float(v).is_integer() else f"{v:.4f}"))

## 6. Fixtures and final fit
Fixtures come from the FPL API, with kickoff times converted to WIB. If the API is down, the notebook uses `FALLBACK_FIXTURES` instead.

If you get an "unknown team" error, the FPL name doesn't match the football-data name. Add the pair to `FPL_TO_FD` in the setup cell.

In [ ]:
def fetch_fixtures(mw):
    try:
        teams = {t["id"]: t["name"] for t in http_get(FPL_BOOTSTRAP_URL).json()["teams"]}
        rows = []
        for f in http_get(FPL_FIXTURES_URL).json():
            if f.get("event") != mw:
                continue
            kickoff = f.get("kickoff_time")
            rows.append(
                {
                    "Kickoff_WIB": pd.to_datetime(kickoff, utc=True).tz_convert(TZ).strftime("%a %d %b %H:%M") if kickoff else "",
                    "Home": FPL_TO_FD.get(teams[f["team_h"]], teams[f["team_h"]]),
                    "Away": FPL_TO_FD.get(teams[f["team_a"]], teams[f["team_a"]]),
                    "Actual": f"{f['team_h_score']}-{f['team_a_score']}" if f.get("finished") else "",
                }
            )
        if rows:
            log.info("Got %d fixtures for MW%d from the FPL API", len(rows), mw)
            return pd.DataFrame(rows)
        log.warning("FPL API returned no fixtures for MW%d", mw)
    except Exception as e:
        log.warning("Could not get fixtures from the FPL API: %s", e)
    if mw not in FALLBACK_FIXTURES:
        raise RuntimeError(f"No fixtures for MW{mw}. Add them to FALLBACK_FIXTURES by hand.")
    log.info("Using FALLBACK_FIXTURES for MW%d", mw)
    return pd.DataFrame([{"Kickoff_WIB": "", "Home": h, "Away": a, "Actual": ""} for h, a in FALLBACK_FIXTURES[mw]])


fixtures = fetch_fixtures(MATCHWEEK)
ref_date = pd.Timestamp(datetime.now(timezone.utc).date())
t0 = time.time()
model = DixonColes().fit(matches[matches.Date < ref_date], ref_date)
log.info(
    "Model fitted in %.1fs | home_adv=x%.3f rho=%.3f offset_E1=x%.3f",
    time.time() - t0, np.exp(model.home), model.rho, np.exp(model.off),
)

rows, matrices = [], {}
for fx in fixtures.itertuples(index=False):
    missing = [t for t in (fx.Home, fx.Away) if t not in model.idx]
    if missing:
        log.error("Unknown team %s. Add it to FPL_TO_FD. Known teams: %s", missing, ", ".join(model.teams))
        continue
    mat, lam, mu = model.score_matrix(fx.Home, fx.Away)
    matrices[f"{fx.Home} vs {fx.Away}"] = mat
    rows.append({**fx._asdict(), **summarize(mat, lam, mu)})
pred = pd.DataFrame(rows)
pred.round(3).to_csv(OUT_DIR / f"predictions_mw{MATCHWEEK}.csv", index=False)
log.info("Saved %d predictions to %s", len(pred), OUT_DIR / f"predictions_mw{MATCHWEEK}.csv")

## 7. Predictions

In [ ]:
cols = ["Kickoff_WIB", "Home", "Away", "xG_Home", "xG_Away", "P_Home", "P_Draw", "P_Away", "Most_Likely", "Top3_Scores", "Over2.5", "BTTS"]
if pred["Actual"].astype(bool).any():
    cols.append("Actual")
if not pred["Kickoff_WIB"].astype(bool).any():
    cols.remove("Kickoff_WIB")
display(
    pred[cols]
    .style.format({"xG_Home": "{:.2f}", "xG_Away": "{:.2f}", "P_Home": "{:.1%}", "P_Draw": "{:.1%}", "P_Away": "{:.1%}", "Over2.5": "{:.1%}", "BTTS": "{:.1%}"})
    .hide(axis="index")
)

### Win / draw / loss

In [ ]:
COLOR_HOME = "#2a78d6"
COLOR_DRAW = "#a8a7a2"
COLOR_AWAY = "#eb6834"

labels = (pred.Home + " vs " + pred.Away).tolist()[::-1]
fig = go.Figure()
for name, col, color, font in (
    ("Home win", "P_Home", COLOR_HOME, "white"),
    ("Draw", "P_Draw", COLOR_DRAW, "#0b0b0b"),
    ("Away win", "P_Away", COLOR_AWAY, "white"),
):
    vals = pred[col].tolist()[::-1]
    fig.add_bar(
        y=labels, x=vals, name=name, orientation="h",
        marker=dict(color=color, line=dict(color="white", width=2)),
        text=[f"{v:.0%}" if v >= 0.08 else "" for v in vals],
        textposition="inside", insidetextanchor="middle", textfont=dict(color=font, size=12),
        hovertemplate="<b>%{y}</b><br>" + name + ": %{x:.1%}<extra></extra>",
    )
fig.update_layout(
    barmode="stack", template="plotly_white", height=90 + 42 * len(labels),
    title=dict(text=f"Matchweek {MATCHWEEK} outcome probabilities", x=0.01),
    xaxis=dict(tickformat=".0%", range=[0, 1.005], dtick=0.2, showgrid=False, zeroline=False),
    yaxis=dict(showgrid=False),
    legend=dict(orientation="h", traceorder="normal", yanchor="bottom", y=1.0, xanchor="left", x=0),
    margin=dict(l=10, r=10, t=80, b=30),
)
fig.show()

### Scoreline probabilities
Pick a match from the dropdown. Rows are home goals, columns are away goals. Only cells at 2% or higher get a label.

In [ ]:
SHOW = 7
BLUE_RAMP = [
    [0.0, "#f4f8fe"], [0.15, "#cde2fb"], [0.3, "#9ec5f4"], [0.45, "#6da7ec"],
    [0.6, "#3987e5"], [0.75, "#256abf"], [0.9, "#184f95"], [1.0, "#0d366b"],
]


def cell_text(z):
    return [[f"{v:.1f}%" if v >= 2 else "" for v in row] for row in z]


names = list(matrices)
zs = [matrices[k][:SHOW, :SHOW] * 100 for k in names]
zmax = max(z.max() for z in zs)
first_home, first_away = names[0].split(" vs ")

heat = go.Figure(
    go.Heatmap(
        z=zs[0], x=list(range(SHOW)), y=list(range(SHOW)),
        text=cell_text(zs[0]), texttemplate="%{text}", textfont=dict(size=12),
        colorscale=BLUE_RAMP, zmin=0, zmax=zmax, xgap=2, ygap=2,
        colorbar=dict(title="%", ticksuffix="%"),
        hovertemplate="Score %{y}-%{x}<br>Probability: %{z:.2f}%<extra></extra>",
    )
)
buttons = []
for k, z in zip(names, zs):
    home, away = k.split(" vs ")
    buttons.append(
        dict(
            label=k, method="update",
            args=[
                {"z": [z], "text": [cell_text(z)]},
                {"title.text": f"Scoreline probabilities: {k}", "yaxis.title.text": f"{home} goals", "xaxis.title.text": f"{away} goals"},
            ],
        )
    )
heat.update_layout(
    template="plotly_white", height=560, width=640,
    title=dict(text=f"Scoreline probabilities: {names[0]}", x=0.01, y=0.97),
    xaxis=dict(title=f"{first_away} goals", side="top", dtick=1, showgrid=False),
    yaxis=dict(title=f"{first_home} goals", autorange="reversed", dtick=1, showgrid=False),
    updatemenus=[dict(buttons=buttons, direction="down", x=0, xanchor="left", y=1.2, yanchor="top", showactive=True)],
    margin=dict(l=10, r=10, t=130, b=10),
)
heat.show()

## 8. Team ratings
Attack above 1 means a team scores more than average. Defence below 1 means it concedes less than average. Rating is attack divided by defence.

Worth a quick look as a sanity check. If the order looks off, try a different `XI` or `L2`.

In [ ]:
epl_teams = set(matches.loc[(matches.Season == CURRENT_SEASON) & (matches.League == "E0"), "HomeTeam"]) | set(fixtures.Home) | set(fixtures.Away)
ratings = model.ratings()
ratings = ratings[ratings.Team.isin(epl_teams)].reset_index(drop=True)
ratings.index = ratings.index + 1
ratings.round(3).to_csv(OUT_DIR / "team_ratings.csv", index_label="Rank")
display(ratings.style.format({"Attack": "{:.3f}", "Defence": "{:.3f}", "Rating": "{:.3f}"}))

## 9. Download
Zips everything in `output/` (predictions, team ratings, backtest detail) and downloads it.

In [ ]:
archive = shutil.make_archive(str(BASE_DIR / f"results_mw{MATCHWEEK}"), "zip", OUT_DIR)
log.info("Archive: %s", archive)
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    log.info("Not running on Colab, file saved at %s", Path(archive).resolve())